[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notes/notebooks/08-pca.ipynb)


# Machine Learning: PCA — Análisis de Componentes Principales

## Colab completo con datos reales, métricas, gráficos, interpretación y comparación predictiva

### Objetivo general

Aprender a aplicar e interpretar **PCA (Principal Component Analysis)** utilizando el mismo dataset real de cáncer de mama de los ejercicios anteriores.

### Al finalizar podrás:

1. Entender qué es reducción de dimensionalidad.
2. Comprender qué es PCA.
3. Diferenciar PCA de selección de variables.
4. Entender por qué el escalamiento es fundamental.
5. Interpretar componentes principales.
6. Analizar varianza explicada y acumulada.
7. Elegir cuántos componentes conservar.
8. Visualizar los datos en 2D y 3D.
9. Interpretar **loadings**.
10. Identificar qué variables contribuyen más a cada componente.
11. Reconstruir los datos y medir pérdida de información.
12. Utilizar PCA dentro de un `Pipeline`.
13. Comparar un modelo **con PCA vs sin PCA**.
14. Evitar *data leakage*.

> PCA es una técnica no supervisada: no utiliza `y` para construir los componentes.



# 1. ¿Qué es reducción de dimensionalidad?

Muchos datasets contienen gran cantidad de variables.

Ejemplos:

- 30 variables clínicas;
- cientos de sensores;
- miles de genes;
- miles de píxeles.

Demasiadas dimensiones pueden producir:

- redundancia;
- variables altamente correlacionadas;
- mayor costo computacional;
- ruido;
- dificultad para visualizar.

La reducción de dimensionalidad intenta representar la información usando menos dimensiones.



# 2. ¿Qué es PCA?

**PCA — Principal Component Analysis** transforma las variables originales en nuevas variables denominadas:

> **Componentes principales**

Cada componente es una combinación lineal de las variables originales.

PCA busca que:

- PC1 capture la mayor variabilidad posible;
- PC2 capture la mayor variabilidad restante;
- PC3 continúe con la siguiente;
- los componentes sean ortogonales entre sí.

En forma conceptual:

\[
PC_1 = a_1X_1+a_2X_2+\cdots+a_pX_p
\]



# 3. PCA vs selección de variables

## Selección de variables

Conserva variables originales.

Ejemplo:

`radius`, `texture`, `area`

## PCA

Crea variables nuevas:

`PC1`, `PC2`, `PC3`

Cada componente combina información de varias variables originales.

### Consecuencia

PCA puede reducir dimensionalidad de forma muy potente, pero normalmente reduce interpretabilidad.



# 4. ¿Para qué se utiliza PCA?

PCA puede utilizarse para:

- visualizar datos de muchas dimensiones;
- reducir redundancia;
- reducir multicolinealidad;
- acelerar modelos;
- comprimir información;
- reducir ruido;
- explorar estructura de datos;
- preparar datos para clustering;
- trabajar con datos genómicos, imágenes o sensores.



# 5. Dataset real: Breast Cancer Wisconsin

Utilizaremos exactamente el mismo dataset:

- **569 observaciones**
- **30 variables predictoras**
- clases conocidas:
  - malignant
  - benign

### Importante

PCA se construirá únicamente a partir de `X`.

La variable `target` se usará después solo para:

- colorear visualizaciones;
- evaluar si la representación PCA conserva información útil para clasificación.


In [ ]:

# Librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:

# Cargar datos
data = load_breast_cancer(as_frame=True)

X = data.data.copy()
y = data.target.copy()

df = X.copy()
df["target"] = y

print("Dimensiones:", df.shape)
print("Variables predictoras:", X.shape[1])
print("Clases:", dict(enumerate(data.target_names)))

display(df.head())


# 6. Exploración inicial — EDA


In [ ]:

print("Valores faltantes totales:", X.isna().sum().sum())

print("\nResumen estadístico:")
display(X.describe().T.head(15))



# 7. ¿Por qué debemos escalar antes de PCA?

PCA depende de la **varianza**.

Si una variable se mueve entre 0 y 1 y otra entre 0 y 100 000, la segunda puede dominar el análisis únicamente por su escala.

Por eso normalmente utilizamos:

`StandardScaler`

Después del escalamiento:

- media ≈ 0;
- desviación estándar ≈ 1.


In [ ]:

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

display(X_scaled_df.head())


In [ ]:

verificacion = pd.DataFrame({
    "Media": X_scaled_df.mean(),
    "Desviación": X_scaled_df.std(ddof=0)
})

display(verificacion.head(10).round(3))



# 8. PCA con todos los componentes

Primero no limitaremos `n_components`.

Así podremos estudiar:

- varianza explicada por cada componente;
- varianza acumulada;
- número de componentes necesarios para diferentes umbrales.


In [ ]:

pca_full = PCA()

X_pca_full = pca_full.fit_transform(X_scaled)

print("Número de componentes:", pca_full.n_components_)



# 9. Varianza explicada

`explained_variance_ratio_` indica qué proporción de la variabilidad total explica cada componente.

### Regla

Cuanto mayor sea el porcentaje de un componente, mayor cantidad de variabilidad resume.


In [ ]:

explained_variance = pca_full.explained_variance_ratio_

componentes = np.arange(
    1,
    len(explained_variance) + 1
)

varianza_acumulada = np.cumsum(
    explained_variance
)

tabla_varianza = pd.DataFrame({
    "Componente": componentes,
    "Varianza_explicada_%": explained_variance * 100,
    "Varianza_acumulada_%": varianza_acumulada * 100
})

display(tabla_varianza.round(3))


## Gráfica de los componentes principales

Esta gráfica sigue la misma lógica del Colab de referencia.

Cada barra representa un **componente principal** y su altura indica el porcentaje de varianza que explica.

Por ejemplo:

- una barra alta en `PC1` significa que el primer componente resume una gran parte de la variabilidad;
- las siguientes barras muestran la información adicional aportada por `PC2`, `PC3`, etc.;
- conforme avanzamos, normalmente cada componente aporta menos información.

El porcentaje se muestra directamente encima de cada barra para facilitar la interpretación.


In [ ]:
# Gráfico de los componentes principales
# Cada barra muestra qué porcentaje de varianza explica cada componente.

porcentajes = explained_variance * 100

plt.figure(figsize=(14, 6))

plt.bar(
    componentes,
    porcentajes
)

plt.title(
    "Varianza explicada por cada componente principal"
)
plt.xlabel("Componentes principales")
plt.ylabel("Varianza explicada (%)")
plt.xticks(componentes)

# Agregar el porcentaje encima de cada barra
for i, porcentaje in enumerate(porcentajes):
    plt.text(
        componentes[i],
        porcentaje + 0.35,
        f"{porcentaje:.2f}%",
        ha="center",
        va="bottom",
        fontsize=7,
        rotation=90
    )

plt.ylim(
    0,
    porcentajes.max() + 8
)

plt.tight_layout()
plt.show()


# 10. Varianza explicada acumulada


In [ ]:

plt.figure(figsize=(10,5))

plt.plot(
    componentes,
    varianza_acumulada * 100,
    marker="o"
)

plt.axhline(80, linestyle="--", label="80%")
plt.axhline(90, linestyle="--", label="90%")
plt.axhline(95, linestyle="--", label="95%")

plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada (%)")
plt.title("Varianza explicada acumulada")
plt.xticks(componentes)
plt.legend()
plt.grid(alpha=0.2)
plt.show()



# 11. ¿Cuántos componentes conservar?

No existe una única regla.

Umbrales frecuentes:

- 80%
- 90%
- PC1 + PC2
- 99%

Conservar más varianza implica:

- más información;
- pero menor reducción dimensional.


In [ ]:

umbrales = [0.80, 0.90, 0.95, 0.99]

resultados = []

for umbral in umbrales:

    n = (
        np.argmax(
            varianza_acumulada >= umbral
        ) + 1
    )

    resultados.append({
        "Varianza_objetivo": f"{int(umbral*100)}%",
        "Componentes": n,
        "Reducción_dimensional_%": (
            1 - n / X.shape[1]
        ) * 100
    })

tabla_umbrales = pd.DataFrame(resultados)

display(tabla_umbrales.round(2))


# 12. PCA con 2 componentes principales

A partir de este punto trabajaremos con **exactamente 2 componentes principales: PC1 y PC2**.

Esto permite:

- reducir las 30 variables originales a solo 2 dimensiones;
- visualizar los datos directamente;
- estudiar cuánta varianza conservan PC1 y PC2;
- utilizar esas dos componentes como entrada para un modelo de Machine Learning.

> Usar 2 componentes produce una reducción dimensional mucho mayor, pero también implica perder más información que si conserváramos PC1 + PC2 de la varianza.


In [ ]:
# PCA principal con exactamente 2 componentes
pca_2 = PCA(
    n_components=2
)

X_pca_2 = pca_2.fit_transform(
    X_scaled
)

print("Dimensión original:", X_scaled.shape)
print("Dimensión después de PCA:", X_pca_2.shape)

print(
    "Varianza explicada por PC1:",
    round(
        pca_2.explained_variance_ratio_[0] * 100,
        2
    ),
    "%"
)

print(
    "Varianza explicada por PC2:",
    round(
        pca_2.explained_variance_ratio_[1] * 100,
        2
    ),
    "%"
)

print(
    "Varianza acumulada PC1 + PC2:",
    round(
        pca_2.explained_variance_ratio_.sum() * 100,
        2
    ),
    "%"
)



# 13. PCA en 2 dimensiones

Una gran utilidad de PCA es visualizar datos de alta dimensión.

Crearemos PC1 y PC2 únicamente para exploración visual.


In [ ]:
# DataFrame con los dos componentes principales
df_pca_2 = pd.DataFrame({
    "PC1": X_pca_2[:, 0],
    "PC2": X_pca_2[:, 1],
    "Clase": y.to_numpy()
})

print(
    "Varianza explicada por PC1 + PC2:",
    round(
        pca_2.explained_variance_ratio_.sum() * 100,
        2
    ),
    "%"
)


In [ ]:

plt.figure(figsize=(9,7))

for clase in sorted(df_pca_2["Clase"].unique()):

    subset = df_pca_2[
        df_pca_2["Clase"] == clase
    ]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        alpha=0.65,
        label=data.target_names[clase]
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA en dos dimensiones")
plt.legend()
plt.grid(alpha=0.2)
plt.show()



### Importante

Las etiquetas malignant/benign se utilizan únicamente para colorear la visualización.

PCA no utilizó esas etiquetas para encontrar PC1 y PC2.


# 14. PCA en 3 dimensiones


In [ ]:

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

pca_3 = PCA(
    n_components=3
)

X_pca_3 = pca_3.fit_transform(
    X_scaled
)

print(
    "Varianza explicada por PC1 + PC2 + PC3:",
    round(
        pca_3.explained_variance_ratio_.sum() * 100,
        2
    ),
    "%"
)

fig = plt.figure(figsize=(10,8))

ax = fig.add_subplot(
    111,
    projection="3d"
)

for clase in np.unique(y):

    mask = (
        y.to_numpy()
        == clase
    )

    ax.scatter(
        X_pca_3[mask, 0],
        X_pca_3[mask, 1],
        X_pca_3[mask, 2],
        alpha=0.6,
        label=data.target_names[clase]
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("PCA en tres dimensiones")
ax.legend()

plt.show()



# 15. ¿Qué son los loadings?

Los **loadings** indican cuánto contribuye cada variable original a un componente.

Un loading grande en valor absoluto indica mayor contribución.

El signo indica dirección, no calidad.

- positivo ≠ bueno;
- negativo ≠ malo.


In [ ]:

loadings = pd.DataFrame(
    pca_full.components_.T,
    index=X.columns,
    columns=[
        f"PC{i}"
        for i in range(
            1,
            pca_full.n_components_ + 1
        )
    ]
)

display(
    loadings[
        ["PC1", "PC2", "PC3"]
    ].head(15).round(4)
)


# 16. Variables que más contribuyen a PC1


In [ ]:

top_pc1 = (
    loadings["PC1"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

tabla_pc1 = pd.DataFrame({
    "Loading": loadings.loc[
        top_pc1.index,
        "PC1"
    ],
    "|Loading|": top_pc1
})

display(tabla_pc1.round(4))


In [ ]:

plt.figure(figsize=(10,6))

plt.barh(
    tabla_pc1.index[::-1],
    tabla_pc1["|Loading|"][::-1]
)

plt.xlabel("|Loading|")
plt.title("Variables con mayor contribución a PC1")
plt.tight_layout()
plt.show()


# 17. Variables que más contribuyen a PC2


In [ ]:

top_pc2 = (
    loadings["PC2"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

tabla_pc2 = pd.DataFrame({
    "Loading": loadings.loc[
        top_pc2.index,
        "PC2"
    ],
    "|Loading|": top_pc2
})

display(tabla_pc2.round(4))


In [ ]:

plt.figure(figsize=(10,6))

plt.barh(
    tabla_pc2.index[::-1],
    tabla_pc2["|Loading|"][::-1]
)

plt.xlabel("|Loading|")
plt.title("Variables con mayor contribución a PC2")
plt.tight_layout()
plt.show()


# 18. Heatmap de loadings


In [ ]:

primeros_componentes = [
    "PC1",
    "PC2",
    "PC3",
    "PC4",
    "PC5"
]

loadings_heatmap = (
    loadings[
        primeros_componentes
    ]
)

plt.figure(figsize=(10,12))

img = plt.imshow(
    loadings_heatmap.values,
    aspect="auto"
)

plt.colorbar(
    img,
    label="Loading"
)

plt.xticks(
    range(
        len(primeros_componentes)
    ),
    primeros_componentes
)

plt.yticks(
    range(
        len(loadings_heatmap.index)
    ),
    loadings_heatmap.index
)

plt.title("Loadings de los primeros componentes")
plt.tight_layout()
plt.show()



# 19. Los componentes son ortogonales

Los componentes principales son construidos para no estar correlacionados linealmente entre sí.

Vamos a comprobarlo con los primeros cinco componentes.


In [ ]:

df_componentes = pd.DataFrame(
    X_pca_full[:, :5],
    columns=[
        "PC1",
        "PC2",
        "PC3",
        "PC4",
        "PC5"
    ]
)

correlacion_componentes = (
    df_componentes.corr()
)

display(
    correlacion_componentes.round(4)
)


In [ ]:

plt.figure(figsize=(7,6))

img = plt.imshow(
    correlacion_componentes.values,
    aspect="auto"
)

plt.colorbar(
    img,
    label="Correlación"
)

plt.xticks(
    range(5),
    correlacion_componentes.columns
)

plt.yticks(
    range(5),
    correlacion_componentes.index
)

plt.title("Correlación entre componentes")
plt.tight_layout()
plt.show()



# 20. Reconstrucción de los datos

PCA permite reconstruir aproximadamente las variables originales.

Con pocos componentes:

- mayor compresión;
- mayor pérdida.

Con más componentes:

- menor pérdida;
- menor reducción.


In [ ]:

def evaluar_reconstruccion(
    X_original,
    n_components
):

    pca = PCA(
        n_components=n_components
    )

    X_reducido = pca.fit_transform(
        X_original
    )

    X_reconstruido = pca.inverse_transform(
        X_reducido
    )

    mse = np.mean(
        (
            X_original
            - X_reconstruido
        ) ** 2
    )

    varianza = (
        pca.explained_variance_ratio_
        .sum()
    )

    return mse, varianza


### Comparación opcional del error de reconstrucción

Aunque el **PCA principal de este notebook utiliza 2 componentes**, aquí probamos varios números de componentes únicamente para observar cómo cambia:

- el error de reconstrucción;
- la varianza conservada.

Esto ayuda a visualizar el costo de reducir de 30 variables a solo **PC1 y PC2**.

La configuración utilizada posteriormente en Machine Learning seguirá siendo **2 componentes**.


In [ ]:
componentes_prueba = [
    2,
    3,
    5,
    10,
    15,
    20,
    25,
    30
]

resultados_rec = []

for n in componentes_prueba:

    # La función devuelve dos resultados:
    # 1) MSE de reconstrucción
    # 2) Varianza conservada
    mse, varianza = evaluar_reconstruccion(
        X_scaled,
        n
    )

    resultados_rec.append({
        "Componentes": n,
        "MSE_reconstruccion": mse,
        "Varianza_conservada_%": (
            varianza * 100
        )
    })

tabla_reconstruccion = pd.DataFrame(
    resultados_rec
)

display(
    tabla_reconstruccion.round(4)
)


In [ ]:

plt.figure(figsize=(9,5))

plt.plot(
    tabla_reconstruccion["Componentes"],
    tabla_reconstruccion["MSE_reconstruccion"],
    marker="o"
)

plt.xlabel("Número de componentes")
plt.ylabel("MSE de reconstrucción")
plt.title("Componentes vs error de reconstrucción")
plt.grid(alpha=0.2)
plt.show()



# 21. PCA dentro de Machine Learning

Ahora responderemos una pregunta práctica:

> **¿Reducir dimensiones con PCA mantiene el rendimiento predictivo?**

Compararemos:

### Modelo A
`StandardScaler → LogisticRegression`

### Modelo B
`StandardScaler → PCA(2 componentes) → LogisticRegression`

Así podemos medir el impacto real de PCA.



# 22. Train/Test para evaluación

Aquí sí utilizaremos `y` porque entraremos en una tarea supervisada de clasificación.

PCA seguirá siendo ajustado únicamente con `X_train` gracias al Pipeline.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


# 23. Modelo sin PCA


In [ ]:

pipeline_sin_pca = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000,
            random_state=RANDOM_STATE
        )
    )
])

pipeline_sin_pca.fit(
    X_train,
    y_train
)

pred_sin_pca = (
    pipeline_sin_pca.predict(
        X_test
    )
)

proba_sin_pca = (
    pipeline_sin_pca
    .predict_proba(
        X_test
    )[:, 1]
)


# 24. Modelo con PCA (2 componentes)


In [ ]:

pipeline_con_pca = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "pca",
        PCA(
            n_components=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000,
            random_state=RANDOM_STATE
        )
    )
])

pipeline_con_pca.fit(
    X_train,
    y_train
)

pred_con_pca = (
    pipeline_con_pca.predict(
        X_test
    )
)

proba_con_pca = (
    pipeline_con_pca
    .predict_proba(
        X_test
    )[:, 1]
)


# 25. Métricas comunes


In [ ]:

def metricas_clasificacion(
    y_true,
    y_pred,
    y_proba
):

    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro"
        ),
        "Recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro"
        ),
        "F1_macro": f1_score(
            y_true,
            y_pred,
            average="macro"
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            y_proba
        )
    }


In [ ]:

met_sin_pca = metricas_clasificacion(
    y_test,
    pred_sin_pca,
    proba_sin_pca
)

met_con_pca = metricas_clasificacion(
    y_test,
    pred_con_pca,
    proba_con_pca
)

comparacion_modelos = pd.DataFrame({
    "Sin PCA": met_sin_pca,
    "Con PCA (2 componentes)": met_con_pca
}).T

display(
    comparacion_modelos.round(4)
)


# 26. Confirmación: el Pipeline utiliza 2 componentes

En este ejercicio fijamos explícitamente:

`PCA(n_components=2)`

Por lo tanto, el modelo supervisado recibe únicamente **PC1 y PC2**.


In [ ]:
pca_pipeline = (
    pipeline_con_pca
    .named_steps["pca"]
)

print(
    "Variables originales:",
    X.shape[1]
)

print(
    "Componentes utilizados:",
    pca_pipeline.n_components_
)

print(
    "Varianza conservada por PC1 + PC2:",
    round(
        pca_pipeline
        .explained_variance_ratio_
        .sum()
        * 100,
        2
    ),
    "%"
)


# 27. Gráfico comparativo del desempeño


In [ ]:

metricas_plot = [
    "Accuracy",
    "Precision_macro",
    "Recall_macro",
    "F1_macro",
    "ROC_AUC"
]

x = np.arange(
    len(metricas_plot)
)

ancho = 0.35

plt.figure(figsize=(11,6))

plt.bar(
    x - ancho/2,
    comparacion_modelos.loc[
        "Sin PCA",
        metricas_plot
    ],
    width=ancho,
    label="Sin PCA"
)

plt.bar(
    x + ancho/2,
    comparacion_modelos.loc[
        "Con PCA (2 componentes)",
        metricas_plot
    ],
    width=ancho,
    label="Con PCA (2 componentes)"
)

plt.xticks(
    x,
    metricas_plot,
    rotation=20
)

plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Modelo sin PCA vs modelo con PCA")
plt.legend()
plt.tight_layout()
plt.show()


# 28. Validación cruzada: con PCA vs sin PCA


In [ ]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "Accuracy": "accuracy",
    "F1_macro": "f1_macro",
    "ROC_AUC": "roc_auc"
}

cv_sin = cross_validate(
    pipeline_sin_pca,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

cv_con = cross_validate(
    pipeline_con_pca,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

resumen_cv = pd.DataFrame({
    "Sin PCA": {
        "CV_Accuracy": (
            cv_sin["test_Accuracy"]
            .mean()
        ),
        "CV_F1_macro": (
            cv_sin["test_F1_macro"]
            .mean()
        ),
        "CV_ROC_AUC": (
            cv_sin["test_ROC_AUC"]
            .mean()
        )
    },
    "Con PCA (2 componentes)": {
        "CV_Accuracy": (
            cv_con["test_Accuracy"]
            .mean()
        ),
        "CV_F1_macro": (
            cv_con["test_F1_macro"]
            .mean()
        ),
        "CV_ROC_AUC": (
            cv_con["test_ROC_AUC"]
            .mean()
        )
    }
}).T

display(
    resumen_cv.round(4)
)


# 29. Matrices de confusión


In [ ]:

cm_sin = confusion_matrix(
    y_test,
    pred_sin_pca
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_sin,
    display_labels=data.target_names
).plot()

plt.title("Matriz de confusión — Sin PCA")
plt.show()


In [ ]:

cm_con = confusion_matrix(
    y_test,
    pred_con_pca
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_con,
    display_labels=data.target_names
).plot()

plt.title("Matriz de confusión — Con PCA")
plt.show()



# 30. ¿PCA siempre mejora Machine Learning?

**No.**

PCA busca conservar varianza de `X`.

Pero:

> una dirección con mucha varianza no necesariamente es la más útil para predecir `y`.

PCA puede:

- mejorar;
- mantener;
- empeorar;

el rendimiento predictivo.

Por eso debe validarse.



# 31. ¿Cuándo PCA puede ayudar?

Puede ser útil cuando:

- existen muchas variables;
- hay alta correlación;
- necesitamos reducir costo;
- queremos visualizar;
- buscamos reducir ruido;
- existen problemas de multicolinealidad.

Puede ser menos útil cuando:

- hay pocas variables;
- cada variable necesita interpretación directa;
- el modelo ya maneja bien correlaciones;
- la reducción elimina información predictiva.



# 32. PCA y Clustering

PCA puede combinarse con clustering:

`StandardScaler → PCA → K-Means`

Ventajas posibles:

- menos dimensiones;
- menor costo;
- menos ruido;
- visualización más sencilla.

Pero debemos comprobar que los componentes conservan suficiente estructura.



# 33. PCA y Árboles / Random Forest

Los árboles:

- no necesitan escalamiento;
- manejan relaciones no lineales;
- manejan interacciones;
- no sufren la multicolinealidad igual que modelos lineales.

Por eso PCA **no siempre mejora** Árboles o Random Forest.

Además, los componentes dificultan interpretar directamente qué variable original produjo una decisión.



# 34. Errores comunes con PCA

1. No escalar las variables.
2. Elegir componentes arbitrariamente.
3. Usar solo 2 componentes porque “se ve bonito”.
4. Pensar que PC1 es una variable original.
5. Confundir PCA con selección de variables.
6. Ajustar PCA antes de dividir Train/Test.
7. Pensar que PCA siempre mejora el modelo.
8. No revisar varianza acumulada.
9. Ignorar loadings.
10. Ignorar la pérdida de interpretabilidad.



# 35. Flujo recomendado en un proyecto real

**Problema → EDA → Train/Test → escalamiento → PCA ajustado en Train → varianza explicada → elegir componentes → transformar Train/Test → entrenar modelo → evaluar → comparar con modelo sin PCA → decidir si la reducción compensa**



# 36. Conclusiones

En este notebook aprendimos que:

- PCA es no supervisado;
- crea componentes principales;
- los componentes son combinaciones lineales;
- el escalamiento suele ser fundamental;
- la varianza explicada ayuda a elegir componentes;
- PCA permite visualizar datos complejos;
- los loadings permiten interpretar las componentes;
- reducir dimensionalidad implica pérdida controlada de información;
- el error de reconstrucción permite medir esa pérdida;
- PCA debe incluirse dentro de un Pipeline para evitar leakage;
- PCA no necesariamente mejora la predicción.

La pregunta correcta es:

> **¿Cuánta dimensionalidad puedo reducir conservando suficiente información y rendimiento para mi problema?**



# 37. Referencias

- Scikit-learn — PCA  
  https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

- Scikit-learn — Decomposition  
  https://scikit-learn.org/stable/modules/decomposition.html

- Scikit-learn — StandardScaler  
  https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

- Scikit-learn — Pipeline  
  https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

- Scikit-learn — Breast Cancer Wisconsin  
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
